# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
print('hi')

hi


In [2]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [3]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
username = "SeanSunny"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [ ]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [ ]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [ ]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [ ]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


In [ ]:
evaluate(human_pricer, test, size=100)

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [5]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [6]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [7]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [8]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [9]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [10]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/5], Train Loss: 9053.730, Val Loss: 11958.276


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/5], Train Loss: 3753.683, Val Loss: 11400.784


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [3/5], Train Loss: 4887.390, Val Loss: 11127.543


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [4/5], Train Loss: 8835.807, Val Loss: 11463.605


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [5/5], Train Loss: 6292.738, Val Loss: 11720.070


In [11]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [12]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$97 $114 $27 $20 $21 $146 $61 $31 $15 $154 $24 $250 $25 $21 $10 $13 $4 $10 $30 $32 $19 $13 $2 $81 $174 $280 $228 $31 $69 $48 $106 $74 $23 $14 $29 $124 $24 $15 $117 $46 $134 $13 $38 $39 $43 $37 $28 $12 $80 $10 $4 $43 $175 $46 $53 $70 $24 $75 $7 $40 $99 $19 $21 $36 $224 $75 $5 $384 $13 $57 $8 $15 $107 $134 $9 $40 $189 $9 $21 $49 $52 $48 $44 $38 $16 $196 $43 $96 $46 $148 $23 $9 $9 $8 $30 $37 $6 $4 $1 $159 $9 $42 $38 $76 $19 $226 $57 $292 $3 $141 $35 $54 $97 $27 $26 $63 $59 $97 $7 $117 $23 $105 $41 $20 $91 $62 $19 $41 $107 $27 $80 $33 $72 $15 $50 $21 $54 $51 $5 $37 $3 $152 $22 $59 $14 $32 $17 $248 $12 $2 $21 $85 $12 $53 $14 $84 $53 $9 $81 $19 $134 $5 $1 $22 $145 $12 $265 $32 $14 $44 $47 $10 $250 $36 $21 $18 $36 $16 $26 $225 $157 $11 $13 $128 $113 $25 $50 $24 $30 $30 $37 $26 $7 $29 $11 $3 $181 $28 $1 $17 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [5]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [6]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [7]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [8]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [9]:
gpt_4__1_nano(test[0])

'$180'

In [10]:
test[0].price

219.0

In [11]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$19 $134 $30 $20 $20 $80 $6 $65 $11 $520 $363 $20 $30 $16 $1 $8 $41 $5 $40 $31 $54 $26 $65 $75 $182 $254 $205 $5 $151 $64 $60 $15 $10 $60 $5 $169 $90 $31 $6 $13 $165 $55 $25 $105 $120 $0 $12 $13 $75 $52 $23 $105 $125 $10 $247 $14 $8 $20 $52 $3 $86 $28 $41 $40 $21 $9 $90 $295 $25 $44 $16 $8 $130 $1 $25 $21 $76 $0 $8 $0 $30 $3 $15 $74 $11 $10 $32 $44 $0 $1 $13 $15 $5 $19 $2 $78 $1 $7 $120 $325 $20 $3 $7 $19 $51 $32 $10 $350 $1 $49 $0 $286 $49 $78 $24 $180 $5 $5 $44 $47 $24 $511 $50 $16 $50 $10 $5 $151 $59 $89 $129 $13 $35 $5 $55 $0 $55 $10 $78 $62 $16 $100 $70 $12 $114 $43 $15 $340 $15 $18 $3 $144 $2 $10 $1 $129 $101 $41 $30 $5 $211 $17 $7 $3 $140 $7 $752 $25 $5 $5 $0 $2 $80 $8 $32 $101 $3 $57 $56 $23 $546 $35 $150 $1 $100 $3 $73 $7 $20 $2 $15 $99 $15 $11 $40 $70 $10 $30 $21 $1 

In [12]:
# The function for gpt-5-nano

def gpt_5_nano(item):
    response = completion(model="openai/gpt-5-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [13]:
evaluate(gpt_5_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$80 $84 $25 $20 $20 $150 $84 $65 $6 $320 $314 $121 $5 $19 $29 $18 $21 $10 $190 $39 $66 $226 $115 $325 $133 $254 $304 $5 $300 $67 $5 $10 $340 $50 $134 $219 $140 $26 $14 $13 $100 $10 $25 $5 $70 $5 $12 $3 $65 $3 $28 $117 $275 $20 $347 $4 $0 $80 $58 $1 $106 $28 $51 $40 $229 $0 $110 $246 $25 $124 $19 $8 $50 $1 $20 $21 $16 $0 $3 $6 $80 $1 $40 $79 $12 $20 $18 $93 $30 $16 $3 $5 $5 $5 $2 $108 $6 $93 $71 $295 $49 $33 $20 $131 $1 $32 $12 $375 $99 $0 $40 $186 $19 $38 $104 $29 $5 $5 $94 $547 $14 $161 $0 $36 $40 $15 $10 $101 $21 $79 $69 $13 $10 $10 $35 $5 $105 $10 $28 $22 $4 $51 $10 $10 $44 $58 $20 $390 $65 $8 $6 $24 $12 $359 $9 $51 $101 $41 $120 $149 $90 $20 $28 $12 $541 $2 $551 $30 $20 $5 $12 $3 $170 $13 $22 $50 $2 $7 $14 $3 $145 $5 $225 $151 $50 $8 $53 $37 $10 $14 $5 $149 $5 $61 $40 $219 $29 $50 $26 $1 

In [14]:
# The function for gpt-5.4-nano

def gpt_5_4_nano(item):
    response = completion(model="openai/gpt-5.4-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [15]:
evaluate(gpt_5_4_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

$31 
Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs

In [16]:
# The function for gpt-5.4-mini

def gpt_5__4_mini(item):
    response = completion(model="gpt-5.4-mini", messages=messages_for(item), seed=42)
    return response.choices[0].message.content


In [17]:
evaluate(gpt_5__4_mini, test)

  0%|          | 0/200 [00:00<?, ?it/s]


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

$10 
Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

$74 $15 $10 $10 
Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

$150 
Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

$84 
Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: htt